# Análisis de canales del EEG — ¿cuántos tienen información y cuáles nos quedamos?

**Versión optimizada:** verificación de versiones al inicio; la selección de canales es **data-driven** (std/SNR por canal y top-25 de MRMR). La lista de canales críticos del Dr. Robles solo se usa para validación (comparación), nunca para restringir qué canales se analizan. Acepta épocas viejas (`_completo_9s.fif`) y nuevas (`-epo.fif`).

**Objetivo:** antes de decidir si el artículo dice 32 o 36 canales, revisar los archivos `.cnt` crudos y responder dos cosas:

1. **¿Cuántos canales tienen información?** (canales con señal real vs. canales planos/muertos)
2. **¿Cuáles son exactamente los canales que nos quedamos?** (los que el pipeline mete en las épocas)

Este notebook solo **lee** los datos, no modifica nada. Corre en Colab, donde está el Drive.


## Instalación de dependencias

En Colab cada sesión nueva arranca con un entorno limpio, por eso esta celda instala primero las librerías con `!pip install` (versiones fijas del último re-corrido) y después verifica e imprime las versiones instaladas. Si las librerías ya están, pip lo detecta rápido y no las vuelve a bajar.

In [6]:
# ==========================================================
# SETUP: INSTALACIÓN DE LIBRERÍAS (Colab)
# ==========================================================
# En Colab cada sesión nueva arranca con un entorno limpio, por eso este
# notebook instala sus librerías al inicio. Si ya están instaladas, pip
# lo detecta rápido y no las vuelve a bajar. El -q deja la salida corta.
!pip install -q mne==1.13.2 numpy==2.4.6 pandas==2.2.3 matplotlib==3.10.0 edfio==0.4.18

from google.colab import drive
drive.mount('/content/drive')

import os, glob
import sys
import importlib
import numpy as np
import pandas as pd
import mne

# ---- Verificación de versiones (después de instalar) ----
# Versiones de referencia del último re-corrido a 250 Hz (Python 3.13).
print(f"Python: {sys.version.split()[0]} (referencia: 3.13)")
for _nombre, _ref in [('mne', '1.13.2'), ('numpy', '2.4.6'), ('pandas', '2.2.3'), ('matplotlib', '3.10.0'), ('edfio', '0.4.18')]:
    try:
        _mod = importlib.import_module(_nombre)
        print(f"  ✓ {_nombre}: {getattr(_mod, '__version__', 'instalado')} (referencia: {_ref})")
    except ImportError:
        print(f"  ✗ {_nombre}: NO INSTALADO (referencia: {_ref})")
print("\n✓ Verificación de versiones completa.")

# Rutas (las mismas del notebook 01)
CARPETA_BASE = "/content/drive/Othercomputers/Mi portátil/Doctorado/VISUAL_DECODING/Neuroscan_Data"
CARPETA_RAW = os.path.join(CARPETA_BASE, "DATA")
CARPETA_EPOCAS = os.path.join(CARPETA_BASE, "Procesado", "03_Epocas_Completas_9s", "FIF")
CARPETA_LOGS = os.path.join(CARPETA_BASE, "Procesado", "01_Bitacoras")
# Top-25 de MRMR (notebook 02) para el cruce data-driven (si existe)
RUTA_MRMR_TOP25 = os.path.join(CARPETA_BASE, "Procesado", "05_Resultados_FeatureEngine",
                               "Caracteristicas_Seleccionadas_MRMR.txt")

# Regla del pipeline (notebook 01, extraer_epocas): se excluyen solo estos
EXCLUIR = ['EOGL', 'EOGR', 'Trigger']

# Canales críticos del Dr. Robles (10). SOLO para validación/comparación,
# nunca para restringir el análisis. T5/T6 son los nombres clásicos de P7/P8
# (nomenclatura moderna 10-20); en los archivos los canales se llaman P7 y P8.
CANALES_ROBLES = ['O1', 'O2', 'Oz', 'PO1', 'PO2', 'P3', 'Pz', 'P4', 'P7', 'P8']

archivos_cnt = sorted(glob.glob(os.path.join(CARPETA_RAW, "*.cnt")))
print(f"Archivos .cnt encontrados: {len(archivos_cnt)}")
print(f"Carpeta raw: {CARPETA_RAW}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.13.18 (referencia: 3.13)
  ✓ mne: 1.13.2 (referencia: 1.13.2)
  ✓ numpy: 2.4.6 (referencia: 2.4.6)
  ✓ pandas: 2.2.3 (referencia: 2.2.3)
  ✓ matplotlib: 3.10.0 (referencia: 3.10.0)
  ✓ edfio: 0.4.18 (referencia: 0.4.18)

✓ Verificación de versiones completa.
Archivos .cnt encontrados: 62
Carpeta raw: /content/drive/Othercomputers/Mi portátil/Doctorado/VISUAL_DECODING/Neuroscan_Data/DATA


## Parte A — ¿Todos los archivos tienen los mismos canales?

Primero leemos solo el **encabezado** de cada `.cnt` (sin cargar los datos, es rápido) y comparamos la lista de canales entre archivos. Si algún archivo trae canales distintos, aquí sale.


In [7]:
# ==========================================================
# PARTE A: lista de canales por archivo (solo encabezado)
# ==========================================================
conjuntos = {}
for ruta in archivos_cnt:
    raw = mne.io.read_raw_cnt(ruta, preload=False, verbose=False)
    conjuntos[os.path.basename(ruta)] = tuple(raw.ch_names)

# ¿Todos tienen el mismo conjunto de canales?
unicos = set(conjuntos.values())
print(f"Archivos revisados: {len(conjuntos)}")
print(f"Conjuntos de canales distintos: {len(unicos)}")
for i, conj in enumerate(unicos, 1):
    n_archivos = sum(1 for v in conjuntos.values() if v == conj)
    print(f"\n--- Conjunto {i} ({n_archivos} archivos, {len(conj)} canales) ---")
    print(list(conj))

# Archivos que NO usan el conjunto más común (si los hay)
from collections import Counter
mas_comun = Counter(conjuntos.values()).most_common(1)[0][0]
diferentes = [f for f, c in conjuntos.items() if c != mas_comun]
if diferentes:
    print("\n⚠️ Archivos con una lista de canales distinta a la más común:")
    for f in diferentes:
        print("  -", f)
else:
    print("\n✓ Todos los archivos tienen exactamente la misma lista de canales.")


/tmp/ipykernel_901/3689863629.py:6: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(ruta, preload=False, verbose=False)
/tmp/ipykernel_901/3689863629.py:6: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(ruta, preload=False, verbose=False)
/tmp/ipykernel_901/3689863629.py:6: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(ruta, preload=False, verbose=False)
/tmp/ipykernel_901/3689863629.py:6: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(ruta, preload=False, verbose=False)
/tmp/ipykernel_901/3689863629.py:6: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(ruta, preload=False, verbose=False)
/tmp/ipykernel_901/3689863629.py:6: RuntimeWarning:   Could not parse meas date from the header. Setting to No

Archivos revisados: 62
Conjuntos de canales distintos: 1

--- Conjunto 1 (62 archivos, 39 canales) ---
['F7', 'Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F8', 'FT7', 'FC3', 'FCz', 'FC4', 'FT8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'TP7', 'CP3', 'CPz', 'CP4', 'TP8', 'A1', 'P7', 'P3', 'Pz', 'P4', 'P8', 'A2', 'O1', 'Oz', 'O2', 'FT9', 'FT10', 'PO1', 'PO2', 'EOGL', 'EOGR', 'Trigger']

✓ Todos los archivos tienen exactamente la misma lista de canales.


## Parte B — ¿Cuántos canales tienen información?

Ahora cargamos una **muestra** de archivos completos y medimos, canal por canal, si hay señal real. Un canal se considera **plano/muerto** si su desviación estándar es prácticamente cero (no varía). También reportamos la amplitud para detectar canales saturados o con ruido extremo.

Por defecto se analizan los primeros 5 archivos; cambia `N_MUESTRA` si quieres más.


In [8]:
# ==========================================================
# PARTE B: ¿qué canales tienen información? (muestra de archivos)
# ==========================================================
import numpy as np
import pandas as pd  # por si el kernel se reinició tras el setup
N_MUESTRA = 5
UMBRAL_PLANO_UV = 0.1   # std < 0.1 µV se considera plano/muerto

muestra = archivos_cnt[:N_MUESTRA]
resumen = []  # una fila por (archivo, canal)

for ruta in muestra:
    nombre = os.path.basename(ruta)
    raw = mne.io.read_raw_cnt(ruta, preload=True, verbose=False)
    data = raw.get_data() * 1e6  # a microvoltios (µV)
    print(f"\n===== {nombre} | {len(raw.ch_names)} canales | {raw.info['sfreq']} Hz =====")
    for i, ch in enumerate(raw.ch_names):
        std = float(np.std(data[i]))
        amp_pp = float(np.ptp(data[i]))
        plano = std < UMBRAL_PLANO_UV
        # Clasificación por nombre
        if ch in ['EOGL', 'EOGR']:
            tipo = 'EOG'
        elif ch in ['A1', 'A2']:
            tipo = 'referencia (mastoides)'
        elif ch == 'Trigger':
            tipo = 'Trigger'
        elif ch.upper().startswith(('ECG', 'EKG')):
            tipo = 'ECG'
        elif ch.upper().startswith('EMG'):
            tipo = 'EMG'
        else:
            tipo = 'EEG'
        kept = ch not in EXCLUIR
        resumen.append({'archivo': nombre, 'canal': ch, 'tipo': tipo,
                        'std_uV': std, 'amp_pp_uV': amp_pp,
                        'plano': plano, 'lo_usa_el_pipeline': kept})
        marca = 'PLANO ⚠️' if plano else ''
        print(f"  {ch:8s} | {tipo:22s} | std={std:9.3f} µV | pico-pico={amp_pp:9.2f} µV | pipeline: {'SÍ' if kept else 'NO'} {marca}")

df = pd.DataFrame(resumen)
print("\n================ RESUMEN DE LA MUESTRA ================")
print(f"Archivos analizados: {df['archivo'].nunique()}")
print(f"Canales por archivo: {df.groupby('archivo')['canal'].nunique().unique()}")
print(f"Canales con información (std >= {UMBRAL_PLANO_UV} µV) por archivo:")
print(df.groupby('archivo').apply(lambda g: int((~g['plano']).sum()), include_groups=False))
print("\nCanales planos/muertos (en al menos un archivo de la muestra):")
planos = df[df['plano']].groupby('canal')['archivo'].apply(list)
print(planos if len(planos) else "  Ninguno ✓")
print("\nCanales que el pipeline mete en las épocas (por archivo):")
print(df.groupby('archivo').apply(lambda g: int(g['lo_usa_el_pipeline'].sum()), include_groups=False))

# ---- Ranking data-driven por std media (señal por canal) ----
# No es un SNR formal, pero ordena los canales por la magnitud de su señal
# y deja ver cuáles aportan más (y cuáles son sospechosamente planos o ruidosos).
ranking_std = (df[df['tipo'] == 'EEG']
               .groupby('canal')['std_uV'].mean()
               .sort_values(ascending=False))
print("\nRanking de canales EEG por std media (µV) — data-driven:")
print(ranking_std.round(2).to_string())
print(f"\nMayor std media: {ranking_std.index[0]} ({ranking_std.iloc[0]:.2f} µV) | "
      f"Menor std media: {ranking_std.index[-1]} ({ranking_std.iloc[-1]:.2f} µV)")


/tmp/ipykernel_901/2296622595.py:14: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(ruta, preload=True, verbose=False)



===== 01_VPF_C_B1 Data.cnt | 39 canales | 1000.0 Hz =====
  F7       | EEG                    | std=   80.392 µV | pico-pico=  1860.27 µV | pipeline: SÍ 
  Fp1      | EEG                    | std=   98.028 µV | pico-pico=  1065.42 µV | pipeline: SÍ 
  Fp2      | EEG                    | std=   93.649 µV | pico-pico=  1498.88 µV | pipeline: SÍ 
  F3       | EEG                    | std=   45.085 µV | pico-pico=   662.53 µV | pipeline: SÍ 
  Fz       | EEG                    | std=   37.656 µV | pico-pico=   543.03 µV | pipeline: SÍ 
  F4       | EEG                    | std=   39.189 µV | pico-pico=   628.47 µV | pipeline: SÍ 
  F8       | EEG                    | std=   49.114 µV | pico-pico=   785.25 µV | pipeline: SÍ 
  FT7      | EEG                    | std=   35.168 µV | pico-pico=   658.62 µV | pipeline: SÍ 
  FC3      | EEG                    | std=   32.517 µV | pico-pico=   836.77 µV | pipeline: SÍ 
  FCz      | EEG                    | std=   26.356 µV | pico-pico=   528.20 

/tmp/ipykernel_901/2296622595.py:14: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(ruta, preload=True, verbose=False)



===== 01_VPF_C_B2 Data.cnt | 39 canales | 1000.0 Hz =====
  F7       | EEG                    | std=   63.282 µV | pico-pico=  1998.97 µV | pipeline: SÍ 
  Fp1      | EEG                    | std=   78.439 µV | pico-pico=   842.47 µV | pipeline: SÍ 
  Fp2      | EEG                    | std=   65.889 µV | pico-pico=   738.46 µV | pipeline: SÍ 
  F3       | EEG                    | std=   37.963 µV | pico-pico=   594.67 µV | pipeline: SÍ 
  Fz       | EEG                    | std=   30.830 µV | pico-pico=   426.80 µV | pipeline: SÍ 
  F4       | EEG                    | std=   31.035 µV | pico-pico=   548.94 µV | pipeline: SÍ 
  F8       | EEG                    | std=   37.957 µV | pico-pico=   685.02 µV | pipeline: SÍ 
  FT7      | EEG                    | std=   30.042 µV | pico-pico=   596.69 µV | pipeline: SÍ 
  FC3      | EEG                    | std=   28.748 µV | pico-pico=   760.84 µV | pipeline: SÍ 
  FCz      | EEG                    | std=   22.142 µV | pico-pico=   381.96 

/tmp/ipykernel_901/2296622595.py:14: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(ruta, preload=True, verbose=False)



===== 01_VPF_F_B1 Data.cnt | 39 canales | 1000.0 Hz =====
  F7       | EEG                    | std=   41.538 µV | pico-pico=   580.10 µV | pipeline: SÍ 
  Fp1      | EEG                    | std=   68.822 µV | pico-pico=   879.18 µV | pipeline: SÍ 
  Fp2      | EEG                    | std=   55.691 µV | pico-pico=   724.49 µV | pipeline: SÍ 
  F3       | EEG                    | std=   26.924 µV | pico-pico=   435.39 µV | pipeline: SÍ 
  Fz       | EEG                    | std=   22.070 µV | pico-pico=   284.37 µV | pipeline: SÍ 
  F4       | EEG                    | std=   20.949 µV | pico-pico=   322.04 µV | pipeline: SÍ 
  F8       | EEG                    | std=   24.088 µV | pico-pico=   568.18 µV | pipeline: SÍ 
  FT7      | EEG                    | std=   24.401 µV | pico-pico=   448.88 µV | pipeline: SÍ 
  FC3      | EEG                    | std=   20.962 µV | pico-pico=   601.46 µV | pipeline: SÍ 
  FCz      | EEG                    | std=   17.080 µV | pico-pico=   240.85 

/tmp/ipykernel_901/2296622595.py:14: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(ruta, preload=True, verbose=False)



===== 01_VPF_F_B2 Data.cnt | 39 canales | 1000.0 Hz =====
  F7       | EEG                    | std=   38.125 µV | pico-pico=   516.23 µV | pipeline: SÍ 
  Fp1      | EEG                    | std=   69.069 µV | pico-pico=   813.33 µV | pipeline: SÍ 
  Fp2      | EEG                    | std=   55.756 µV | pico-pico=   714.99 µV | pipeline: SÍ 
  F3       | EEG                    | std=   26.302 µV | pico-pico=   452.67 µV | pipeline: SÍ 
  Fz       | EEG                    | std=   22.655 µV | pico-pico=   398.71 µV | pipeline: SÍ 
  F4       | EEG                    | std=   22.498 µV | pico-pico=   589.88 µV | pipeline: SÍ 
  F8       | EEG                    | std=   26.961 µV | pico-pico=   624.86 µV | pipeline: SÍ 
  FT7      | EEG                    | std=   22.657 µV | pico-pico=   328.78 µV | pipeline: SÍ 
  FC3      | EEG                    | std=   20.670 µV | pico-pico=   527.95 µV | pipeline: SÍ 
  FCz      | EEG                    | std=   17.483 µV | pico-pico=   406.65 

/tmp/ipykernel_901/2296622595.py:14: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw = mne.io.read_raw_cnt(ruta, preload=True, verbose=False)



===== 02_SUM_C_A1 Data.cnt | 39 canales | 1000.0 Hz =====
  F7       | EEG                    | std=   21.477 µV | pico-pico=   511.76 µV | pipeline: SÍ 
  Fp1      | EEG                    | std=   47.985 µV | pico-pico=   573.57 µV | pipeline: SÍ 
  Fp2      | EEG                    | std=   39.985 µV | pico-pico=   493.70 µV | pipeline: SÍ 
  F3       | EEG                    | std=   20.221 µV | pico-pico=   479.99 µV | pipeline: SÍ 
  Fz       | EEG                    | std=   20.180 µV | pico-pico=   353.34 µV | pipeline: SÍ 
  F4       | EEG                    | std=   20.744 µV | pico-pico=   476.29 µV | pipeline: SÍ 
  F8       | EEG                    | std=   28.721 µV | pico-pico=   967.47 µV | pipeline: SÍ 
  FT7      | EEG                    | std=   13.136 µV | pico-pico=   356.99 µV | pipeline: SÍ 
  FC3      | EEG                    | std=   17.029 µV | pico-pico=   510.73 µV | pipeline: SÍ 
  FCz      | EEG                    | std=   16.211 µV | pico-pico=   337.07 

## Parte C — La lista exacta de los que nos quedamos

Aplicamos la regla del pipeline a la lista de canales del `.cnt` y la contrastamos con los canales que realmente quedaron guardados en un archivo de épocas procesado (`.fif`). Así vemos si lo que el código dice que toma coincide con lo que quedó en disco.


In [9]:
# ==========================================================
# PARTE C: canales que nos quedamos (regla del pipeline vs .fif)
# ==========================================================
# 1) Lista según la regla del pipeline, tomada del primer .cnt
raw0 = mne.io.read_raw_cnt(archivos_cnt[0], preload=False, verbose=False)
canales_todos = list(raw0.ch_names)
canales_kept = [ch for ch in canales_todos if ch not in EXCLUIR]
canales_excl = [ch for ch in canales_todos if ch in EXCLUIR]
print(f"Canales totales en el .cnt: {len(canales_todos)}")
print(f"Canales que el pipeline mete en las épocas: {len(canales_kept)}")
print("  ", canales_kept)
print(f"Canales excluidos por la regla: {len(canales_excl)} -> {canales_excl}")

# 2) Lista real guardada en un archivo de épocas procesado
# Acepta los archivos viejos (_completo_9s.fif) y los nuevos (-epo.fif)
fifs = sorted(set(glob.glob(os.path.join(CARPETA_EPOCAS, "*_completo_9s.fif")) +
                  glob.glob(os.path.join(CARPETA_EPOCAS, "*_completo_9s-epo.fif"))))
print(f"\nArchivos de épocas .fif: {len(fifs)}")
if fifs:
    ep = mne.read_epochs(fifs[0], verbose=False)
    # Chequeo de línea base: las épocas del notebook 01 optimizado la traen aplicada
    print(f"Baseline de las épocas cargadas: {ep.baseline} "
          f"({'✓ aplicada' if ep.baseline is not None else '⚠️ NO aplicada (ver notebook 01/02)'})")
    print(f"Ejemplo: {os.path.basename(fifs[0])}")
    print(f"Canales en las épocas: {len(ep.ch_names)}")
    print("  ", list(ep.ch_names))
    print(f"¿Coincide con la regla del pipeline? {list(ep.ch_names) == canales_kept}")
    if list(ep.ch_names) != canales_kept:
        print("  En épocas pero no en la regla:", sorted(set(ep.ch_names) - set(canales_kept)))
        print("  En la regla pero no en épocas:", sorted(set(canales_kept) - set(ep.ch_names)))


Canales totales en el .cnt: 39
Canales que el pipeline mete en las épocas: 36
   ['F7', 'Fp1', 'Fp2', 'F3', 'Fz', 'F4', 'F8', 'FT7', 'FC3', 'FCz', 'FC4', 'FT8', 'T7', 'C3', 'Cz', 'C4', 'T8', 'TP7', 'CP3', 'CPz', 'CP4', 'TP8', 'A1', 'P7', 'P3', 'Pz', 'P4', 'P8', 'A2', 'O1', 'Oz', 'O2', 'FT9', 'FT10', 'PO1', 'PO2']
Canales excluidos por la regla: 3 -> ['EOGL', 'EOGR', 'Trigger']

Archivos de épocas .fif: 0


/tmp/ipykernel_901/2486194313.py:5: RuntimeWarning:   Could not parse meas date from the header. Setting to None.
  raw0 = mne.io.read_raw_cnt(archivos_cnt[0], preload=False, verbose=False)


## Resumen final (se guarda como CSV)

Una tabla por canal con: tipo, si tiene información, si el pipeline lo usa, y si es plano en algún archivo. Se guarda en `01_Bitacoras/` para consultarla después.


In [10]:
# ==========================================================
# RESUMEN FINAL: tabla por canal + CSV
# ==========================================================
tabla = (df.groupby('canal')
           .agg(tipo=('tipo', 'first'),
                archivos=('archivo', 'nunique'),
                std_media_uV=('std_uV', 'mean'),
                plano_en_algun_archivo=('plano', 'any'),
                lo_usa_el_pipeline=('lo_usa_el_pipeline', 'first'))
           .reset_index())
tabla['tiene_informacion'] = ~tabla['plano_en_algun_archivo']
tabla = tabla.sort_values(['lo_usa_el_pipeline', 'tipo', 'canal'], ascending=[False, True, True])
print(tabla.to_string(index=False))
print(f"\nTotal de canales: {len(tabla)}")
print(f"Con información (no planos): {int(tabla['tiene_informacion'].sum())}")
print(f"Que el pipeline mete en las épocas: {int(tabla['lo_usa_el_pipeline'].sum())}")
print(f"Con información Y usados por el pipeline: {int((tabla['tiene_informacion'] & tabla['lo_usa_el_pipeline']).sum())}")
print(f"Usados por el pipeline pero planos: {int((~tabla['tiene_informacion'] & tabla['lo_usa_el_pipeline']).sum())}")

# ---- Cruce data-driven con el top-25 de MRMR (notebook 02) ----
# Si existe el archivo de MRMR, marcamos qué canales seleccionó el modelo.
# La selección de canales relevantes es DATA-DRIVEN (MRMR + std); la lista de
# Robles solo se usa para validación (columna 'es_canal_robles').
canales_mrmr = set()
if os.path.exists(RUTA_MRMR_TOP25):
    with open(RUTA_MRMR_TOP25) as f_mrmr:
        feats_mrmr = [l.strip() for l in f_mrmr if l.strip()]
    canales_mrmr = set(fe.split('_')[0] for fe in feats_mrmr)
    print(f"\nTop-25 MRMR cargado ({len(feats_mrmr)} características, {len(canales_mrmr)} canales): {sorted(canales_mrmr)}")
else:
    print(f"\n⚠️ No se encontró el top-25 de MRMR en {RUTA_MRMR_TOP25} (corre el notebook 02 primero).")
tabla['en_top25_MRMR'] = tabla['canal'].isin(canales_mrmr)
tabla['es_canal_robles'] = tabla['canal'].isin(CANALES_ROBLES)
print(f"\nCanales en el top-25 de MRMR: {int(tabla['en_top25_MRMR'].sum())} -> "
      f"{sorted(tabla.loc[tabla['en_top25_MRMR'], 'canal'])}")
print(f"Canales de Robles con información y usados por el pipeline: "
      f"{int((tabla['es_canal_robles'] & tabla['tiene_informacion'] & tabla['lo_usa_el_pipeline']).sum())} -> "
      f"{sorted(tabla.loc[tabla['es_canal_robles'] & tabla['tiene_informacion'], 'canal'])}")
print(f"Canales en MRMR y de Robles (coincidencia): "
      f"{sorted(set(tabla.loc[tabla['en_top25_MRMR'], 'canal']) & set(CANALES_ROBLES))}")

# La carpeta Procesado/01_Bitacoras la crea el notebook 01 al correr;
# si este notebook corre primero, la creamos aquí para poder guardar la tabla.
os.makedirs(CARPETA_LOGS, exist_ok=True)
salida = os.path.join(CARPETA_LOGS, "analisis_canales.csv")
tabla.to_csv(salida, index=False)
print(f"\n✓ Tabla guardada en: {salida}")


  canal                   tipo  archivos  std_media_uV  plano_en_algun_archivo  lo_usa_el_pipeline  tiene_informacion
     C3                    EEG         5     18.170540                   False                True               True
     C4                    EEG         5     16.322772                   False                True               True
    CP3                    EEG         5     14.478087                   False                True               True
    CP4                    EEG         5     13.764934                   False                True               True
    CPz                    EEG         5     13.806780                   False                True               True
     Cz                    EEG         5     16.032484                   False                True               True
     F3                    EEG         5     31.298997                   False                True               True
     F4                    EEG         5     26.882932  